In [131]:
import pandas as pd
import numpy as np
import matplotlib



In [132]:
data = pd.read_csv('Hyderabad_houses_cleaned_DS.csv')
data.head()

,Price,Area,Location,No. of Bedrooms,Resale,MaintenanceStaff,Gymnasium,SwimmingPool,LandscapedGardens,JoggingTrack,...,LiftAvailable,BED,VaastuCompliant,Microwave,GolfCourse,TV,DiningTable,Sofa,Wardrobe,Refrigerator
0,6968000,1340,Nizampet,2,0,0.0,1.0,1.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,29000000,3498,Hitech City,4,0,0.0,1.0,1.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,6590000,1318,Manikonda,2,0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5739000,1295,Alwal,3,1,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5679000,1145,Kukatpally,2,0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [133]:
selected_columns = [
    "Area",
    "Location",
    "No. of Bedrooms",
    "Resale",
    "CarParking",
    "SwimmingPool",
    "JoggingTrack",
    "LiftAvailable",
    "Price"
]

df = data[selected_columns]

df.head()

,Area,Location,No. of Bedrooms,Resale,CarParking,SwimmingPool,JoggingTrack,LiftAvailable,Price
0,1340,Nizampet,2,0,1.0,1.0,1.0,1.0,6968000
1,3498,Hitech City,4,0,1.0,1.0,1.0,1.0,29000000
2,1318,Manikonda,2,0,0.0,0.0,0.0,0.0,6590000
3,1295,Alwal,3,1,0.0,0.0,0.0,1.0,5739000
4,1145,Kukatpally,2,0,1.0,0.0,0.0,1.0,5679000


In [134]:
X = df.drop("Price", axis=1)

y = df["Price"]

In [135]:
X.head()

,Area,Location,No. of Bedrooms,Resale,CarParking,SwimmingPool,JoggingTrack,LiftAvailable
0,1340,Nizampet,2,0,1.0,1.0,1.0,1.0
1,3498,Hitech City,4,0,1.0,1.0,1.0,1.0
2,1318,Manikonda,2,0,0.0,0.0,0.0,0.0
3,1295,Alwal,3,1,0.0,0.0,0.0,1.0
4,1145,Kukatpally,2,0,1.0,0.0,0.0,1.0


In [136]:
y.head()

0     6968000
1    29000000
2     6590000
3     5739000
4     5679000
Name: Price, dtype: int64

In [137]:
#train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [138]:
X_train.shape,X_test.shape

((1946, 8), (487, 8))

*** Prepressing ***

In [139]:
numerical_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("numerical_features :",numerical_features)
print("categorical_features :",categorical_features)

numerical_features : ['Area', 'No. of Bedrooms', 'Resale', 'CarParking', 'SwimmingPool', 'JoggingTrack', 'LiftAvailable']
categorical_features : ['Location']


In [140]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# numerical_features--preprocessing_steps
numerical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# categorical_features--preprocessing_steps
categorical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("OHE", OneHotEncoder(handle_unknown="ignore",sparse_output=False))
    ]
)

# preprocess pipeline
from sklearn.compose import ColumnTransformer
preprocess = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat",categorical_transformer, categorical_features)
    ]
)


Baseline model ((simple lr))

In [141]:
from sklearn.linear_model import LinearRegression

baseline_pipe = Pipeline(
    steps=[
        ("preprocess",preprocess),
        ("model", LinearRegression())
    ]
)


In [142]:
# preprocess the data & train the baseline model
baseline_pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

Evalivation of baseline model

In [143]:
train_baseline_pred = baseline_pipe.predict(X_train)
test_baseline_pred = baseline_pipe.predict(X_test)



In [144]:
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

In [145]:
rmse_train= root_mean_squared_error(y_train,train_baseline_pred)
r2_train = r2_score(y_train,train_baseline_pred)
mae_train = mean_absolute_error(y_train,train_baseline_pred)

rmse_train,r2_train,mae_train

(2704982.7756024664, 0.8923392527647163, 1514475.1408431847)

In [146]:
train_baseline_pred[:5]

array([7940301.77710982, 7121249.5708667 , 5370842.02595192,
       8773625.32803683, 3688516.73336235])

In [147]:
y_train[:5]

2337    10000000
1100     9075000
1526     3500000
298      9760000
927      3600000
Name: Price, dtype: int64

In [148]:
rmse_test= root_mean_squared_error(y_test,test_baseline_pred)
r2_test = r2_score(y_test,test_baseline_pred)
mae_test = mean_absolute_error(y_test, test_baseline_pred)

rmse_test,r2_test,mae_test

(3300723.499810287, 0.8454393262839451, 1778058.528469562)

In [149]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import KFold

RANDOM_STATE = 42

# Models to try
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Lasso": Lasso(random_state=RANDOM_STATE, max_iter=10000),
    "RandomForest": RandomForestRegressor(random_state=RANDOM_STATE),
    "HistGB": HistGradientBoostingRegressor(random_state=RANDOM_STATE)
}

# Cross-validation
k = 5
cv = KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)

# Scoring metrics
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

In [150]:
from sklearn.model_selection import cross_validate

In [151]:
rows = []

for name, model in models.items():

    pipe = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", model)
        ]
    )

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    rows.append({
        "model": name,
        "cv_rmse": -scores["test_rmse"].mean(),
        "cv_mae": -scores["test_mae"].mean(),
        "cv_r2": scores["test_r2"].mean()
    })

# Sort based on lowest RMSE
cv_results = pd.DataFrame(rows).sort_values("cv_rmse")

print("=== CV Model Comparison ===")
print(cv_results)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.696e+14, tolerance: 1.059e+13
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.465e+13, tolerance: 1.109e+13
  model = cd_fast.enet_coordinate_descent(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iter

=== CV Model Comparison ===
              model       cv_rmse        cv_mae     cv_r2
1             Ridge  3.015048e+06  1.754462e+06  0.863962
4            HistGB  3.035583e+06  1.557497e+06  0.863719
3      RandomForest  3.058658e+06  1.350501e+06  0.860639
2             Lasso  3.091900e+06  1.771579e+06  0.857137
0  LinearRegression  3.092905e+06  1.773868e+06  0.857065


In [154]:
scores

{'fit_time': array([0.72442412, 0.69523215, 0.72507095, 0.71828103, 0.70490623]),
 'score_time': array([0.00680399, 0.00825977, 0.00623417, 0.0064528 , 0.0073638 ]),
 'test_rmse': array([-3031696.05202569, -2814514.01259013, -3973912.76470172,
        -2430723.39096945, -2927071.10442763]),
 'test_mae': array([-1683492.55498207, -1535182.70697107, -1759388.15703895,
        -1442756.19394773, -1366665.20534488]),
 'test_r2': array([0.8641997 , 0.85582317, 0.82883086, 0.89217494, 0.87756557])}

In [155]:
rows

[{'model': 'LinearRegression',
  'cv_rmse': np.float64(3092905.2754402375),
  'cv_mae': np.float64(1773868.4135448807),
  'cv_r2': np.float64(0.857065036220914)},
 {'model': 'Ridge',
  'cv_rmse': np.float64(3015047.9854259314),
  'cv_mae': np.float64(1754461.6576831776),
  'cv_r2': np.float64(0.8639615109262806)},
 {'model': 'Lasso',
  'cv_rmse': np.float64(3091899.68105141),
  'cv_mae': np.float64(1771578.53949536),
  'cv_r2': np.float64(0.8571368831696887)},
 {'model': 'RandomForest',
  'cv_rmse': np.float64(3058658.284293972),
  'cv_mae': np.float64(1350501.1307827374),
  'cv_r2': np.float64(0.8606389566780465)},
 {'model': 'HistGB',
  'cv_rmse': np.float64(3035583.464942925),
  'cv_mae': np.float64(1557496.96365694),
  'cv_r2': np.float64(0.8637188479605843)}]

In [156]:
cv_results

,model,cv_rmse,cv_mae,cv_r2
1,Ridge,3.015048e+06,1.754462e+06,0.863962
4,HistGB,3.035583e+06,1.557497e+06,0.863719
3,RandomForest,3.058658e+06,1.350501e+06,0.860639
2,Lasso,3.091900e+06,1.771579e+06,0.857137
0,LinearRegression,3.092905e+06,1.773868e+06,0.857065


In [ ]:
# from sklearn.pipeline import Pipeline
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import GridSearchCV

# # Pipeline
# rf_pipeline = Pipeline(
#     steps=[
#         ("preprocess", preprocess),
#         ("model", RandomForestRegressor(random_state=RANDOM_STATE))
#     ]
# )

# # Parameter Grid
# param_grid = {
#     "model__n_estimators": [100, 200, 300, 500],
#     "model__max_depth": [None, 10, 20, 30, 40],
#     "model__min_samples_split": [2, 5, 10, 20],
#     "model__min_samples_leaf": [1, 2, 4, 8],
#     "model__max_features": ["sqrt", "log2", None],
#     "model__bootstrap": [True, False]
# }

# # Grid Search
# grid_search = GridSearchCV(
#     estimator=rf_pipeline,
#     param_grid=param_grid,
#     cv=5,
#     scoring="neg_root_mean_squared_error",
#     n_jobs=-1,
#     verbose=2,
#     return_train_score=True
# )

# # Fit
# grid_search.fit(X_train, y_train)

# print("Best Parameters:")
# print(grid_search.best_params_)

# print("\nBest CV RMSE:")
# print(-grid_search.best_score_)

# best_rf_model = grid_search.best_estimator_

Fitting 5 folds for each of 1920 candidates, totalling 9600 fits
[CV] END model__bootstrap=True, model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.3s
[CV] END model__bootstrap=True, model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.2s
[CV] END model__bootstrap=True, model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.3s
[CV] END model__bootstrap=True, model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.2s
[CV] END model__bootstrap=True, model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=2, model__n_estimators=100; total time=   0.3s
[CV] END model__bootstrap=True, mod

In [ ]:
# from sklearn.metrics import (
#     mean_squared_error,
#     mean_absolute_error,
#     r2_score
# )
# import numpy as np

# y_pred = best_rf_model.predict(X_test)

# rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# mae = mean_absolute_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)

# print("Test RMSE:", rmse)
# print("Test MAE:", mae)
# print("Test R²:", r2)

Test RMSE: 2839146.21466716
Test MAE: 1228849.8173087013
Test R²: 0.8856447178261722


'model__bootstrap': False, 'model__max_depth': 40, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 100

In [173]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Final Random Forest model with tuned hyperparameters
best_rf_model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", RandomForestRegressor(
            n_estimators=300,
            # max_depth=40,
            # max_features="sqrt",
            # min_samples_leaf=1,
            # min_samples_split=2,
            # bootstrap=False,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]
)

# Train the model
best_rf_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [174]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Predictions
y_pred = best_rf_model.predict(X_test)

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("===== Final Random Forest Results =====")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

===== Final Random Forest Results =====
RMSE : 2405323.5196
MAE  : 1183458.6882
R²   : 0.9179


In [175]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Training predictions
y_train_pred = best_rf_model.predict(X_train)

# Training metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

print("===== TRAINING RESULTS =====")
print(f"RMSE : {train_rmse:.4f}")
print(f"MAE  : {train_mae:.4f}")
print(f"R²   : {train_r2:.4f}")

===== TRAINING RESULTS =====
RMSE : 1110694.1230
MAE  : 512399.5722
R²   : 0.9818


In [176]:
# Test predictions
y_test_pred = best_rf_model.predict(X_test)

# Test metrics
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("\n===== TEST RESULTS =====")
print(f"RMSE : {test_rmse:.4f}")
print(f"MAE  : {test_mae:.4f}")
print(f"R²   : {test_r2:.4f}")


===== TEST RESULTS =====
RMSE : 2405323.5196
MAE  : 1183458.6882
R²   : 0.9179


In [179]:
comparison = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "RMSE": [train_rmse, test_rmse],
    "MAE": [train_mae, test_mae],
    "R2": [train_r2, test_r2]
})

print(comparison)

  Dataset          RMSE           MAE        R2
0   Train  1.110694e+06  5.123996e+05  0.981848
1    Test  2.405324e+06  1.183459e+06  0.917922


In [180]:
import joblib

joblib.dump(best_rf_model, "house_price_model.pkl")

['house_price_model.pkl']